# Benchmark SSVEP - TDCA Paper-Style Evaluation

?? notebook ??? `benchmark` ???? SSVEP ???????? TDCA ?????????

1. Benchmark ?????????? 0.5s?????? 0.14s???????
2. 9 ???????`PZ, PO5, PO3, POZ, PO4, PO6, O1, OZ, O2`?
3. 40 ?????? Benchmark `.mat` ?????????`8,9,...,15, 8.2,9.2,...,15.8`?
4. FBCCA ????????
5. FBTDCA??????????????????????DSP ??????????????
6. ??????????? block ??????? 5 ? block??? 1 ? block?

????? `Python (bci_env)`?`bciwork_1` ????????/????????????? notebook?


In [ ]:
from __future__ import annotations

import math
import warnings
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import signal
from scipy.io import loadmat
from scipy.linalg import eigh, qr, svdvals

warnings.filterwarnings("ignore")
np.set_printoptions(precision=4, suppress=True)


In [ ]:
# =========================
# Global configuration
# =========================
ROOT = Path.cwd().resolve()
BENCHMARK_DIR = ROOT / "benchmark"
if not BENCHMARK_DIR.exists():
    BENCHMARK_DIR = ROOT.parent / "benchmark"
if not BENCHMARK_DIR.exists():
    raise FileNotFoundError("Cannot find benchmark directory. Put this notebook beside E:/???????/benchmark.")

RESULT_DIR = ROOT / "ssvep_results"
RESULT_DIR.mkdir(exist_ok=True)

FS = 250
PRE_STIM_SEC = 0.5
VISUAL_LATENCY_SEC = 0.14
WINDOW_SEC = 4.0
WINDOW_SAMPLES = int(round(WINDOW_SEC * FS))

# TDCA paper-style settings.
NUM_HARMONICS = 5
NUM_FILTER_BANKS = 5
DELAY_POINTS = 5          # include 0..5 shifted copies, about 20 ms at 250 Hz
TDCA_COMPONENTS = 1

# Keep this small for smoke tests. Set to None for all 35 subjects.
SUBJECT_LIMIT = 1
RUN_FBCCA = True
RUN_FBTDCA = True

# Benchmark target dimension order in the .mat files: row-wise 5 x 8 grid.
TARGET_FREQS = np.asarray(
    [base + offset for offset in np.arange(0.0, 1.0, 0.2) for base in np.arange(8.0, 16.0, 1.0)],
    dtype=float,
)
N_CLASSES = len(TARGET_FREQS)

CHANNEL_NAMES_64 = [
    "FP1", "FPZ", "FP2", "AF3", "AF4", "F7", "F5", "F3", "F1", "FZ", "F2", "F4", "F6", "F8",
    "FT7", "FC5", "FC3", "FC1", "FCZ", "FC2", "FC4", "FC6", "FT8", "T7", "C5", "C3", "C1", "CZ", "C2", "C4", "C6", "T8",
    "M1", "TP7", "CP5", "CP3", "CP1", "CPZ", "CP2", "CP4", "CP6", "M2", "TP8", "P7", "P5", "P3", "P1", "PZ", "P2", "P4", "P6", "P8",
    "PO7", "PO5", "PO3", "POZ", "PO4", "PO6", "PO8", "CB1", "O1", "OZ", "O2", "CB2",
]
USE_CHANNELS = ["PZ", "PO5", "PO3", "POZ", "PO4", "PO6", "O1", "OZ", "O2"]
CHANNEL_IDXS = [CHANNEL_NAMES_64.index(ch) for ch in USE_CHANNELS]

print("benchmark:", BENCHMARK_DIR)
print("result dir:", RESULT_DIR)
print("window:", WINDOW_SEC, "s, samples:", WINDOW_SAMPLES, "latency:", VISUAL_LATENCY_SEC)
print("channels:", list(zip(USE_CHANNELS, CHANNEL_IDXS)))
print("freq first/last:", TARGET_FREQS[:8], TARGET_FREQS[-8:])


In [ ]:
def discover_subject_files(data_dir: Path, limit: int | None = None) -> list[tuple[int, Path]]:
    files = []
    for subject_dir in sorted(data_dir.glob("S*.mat"), key=lambda p: int(p.stem[1:])):
        sid = int(subject_dir.stem[1:])
        mat_path = subject_dir / f"S{sid}.mat"
        if mat_path.exists():
            files.append((sid, mat_path))
    if limit is not None:
        files = files[: int(limit)]
    if not files:
        raise FileNotFoundError(f"No subject files found under {data_dir}")
    return files

subject_files = discover_subject_files(BENCHMARK_DIR, SUBJECT_LIMIT)
print("subjects:", [(sid, str(path)) for sid, path in subject_files[:5]], "count=", len(subject_files))
sample_data = loadmat(subject_files[0][1])["data"]
print("sample data shape:", sample_data.shape, "= channels x samples x targets x blocks")


In [ ]:
def load_subject_data(mat_path: Path) -> np.ndarray:
    data = np.asarray(loadmat(mat_path)["data"], dtype=np.float64)
    if data.shape[0] != 64 or data.shape[2] != N_CLASSES:
        raise ValueError(f"Unexpected Benchmark data shape: {data.shape}")
    return data


def extract_trials(data: np.ndarray, extra_points: int = 0) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Return X, y, block_id. X shape = trials x channels x samples."""
    start = int(round((PRE_STIM_SEC + VISUAL_LATENCY_SEC) * FS))
    stop = start + WINDOW_SAMPLES + int(extra_points)
    if stop > data.shape[1]:
        raise ValueError(f"Requested [{start}:{stop}] but data has only {data.shape[1]} samples")

    n_blocks = data.shape[3]
    xs, ys, blocks = [], [], []
    for block in range(n_blocks):
        for target in range(N_CLASSES):
            trial = data[CHANNEL_IDXS, start:stop, target, block]
            xs.append(trial)
            ys.append(target)
            blocks.append(block)
    x = np.stack(xs, axis=0)
    y = np.asarray(ys, dtype=int)
    block_ids = np.asarray(blocks, dtype=int)
    return x, y, block_ids


def zscore_epoch(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float64)
    return x - x.mean(axis=-1, keepdims=True)


In [ ]:
def make_reference_bank(freqs: np.ndarray, n_samples: int, fs: int = FS, n_harmonics: int = NUM_HARMONICS) -> np.ndarray:
    t = np.arange(n_samples, dtype=np.float64) / fs
    refs = []
    for freq in freqs:
        rows = []
        for harmonic in range(1, n_harmonics + 1):
            rows.append(np.sin(2 * np.pi * harmonic * freq * t))
            rows.append(np.cos(2 * np.pi * harmonic * freq * t))
        refs.append(np.asarray(rows, dtype=np.float64))
    return np.asarray(refs, dtype=np.float64)


def reference_projectors(refs: np.ndarray) -> list[np.ndarray]:
    projectors = []
    for ref in refs:
        q, _ = qr(ref.T, mode="economic")
        projectors.append(q @ q.T)
    return projectors

REFS = make_reference_bank(TARGET_FREQS, WINDOW_SAMPLES)
PROJECTORS = reference_projectors(REFS)
print("refs:", REFS.shape, "projector:", PROJECTORS[0].shape)


In [ ]:
def cca_score_one(epoch: np.ndarray, refs: np.ndarray = REFS) -> np.ndarray:
    x = zscore_epoch(epoch[:, :WINDOW_SAMPLES])
    qx, _ = qr(x.T, mode="economic")
    scores = np.empty(refs.shape[0], dtype=np.float64)
    for i, ref in enumerate(refs):
        qy, _ = qr(ref.T, mode="economic")
        s = svdvals(qx.T @ qy)
        scores[i] = s[0] if s.size else 0.0
    return scores


def predict_cca(x_test: np.ndarray) -> np.ndarray:
    return np.asarray([int(np.argmax(cca_score_one(epoch))) for epoch in x_test], dtype=int)


def design_filter_bank(n_bands: int = NUM_FILTER_BANKS, fs: int = FS) -> list[np.ndarray]:
    # Common SSVEP filter-bank setting used by FBCCA/TRCA/TDCA papers.
    pass_low = np.asarray([6, 14, 22, 30, 38, 46, 54, 62, 70, 78], dtype=float)
    stop_low = np.asarray([4, 10, 16, 24, 32, 40, 48, 56, 64, 72], dtype=float)
    nyq = fs / 2.0
    high_pass = min(90.0, nyq - 5.0)
    high_stop = min(100.0, nyq - 2.0)
    filters = []
    for band in range(n_bands):
        wp = [pass_low[band] / nyq, high_pass / nyq]
        ws = [stop_low[band] / nyq, high_stop / nyq]
        order, wn = signal.cheb1ord(wp, ws, gpass=3, gstop=40)
        sos = signal.cheby1(order, rp=0.5, Wn=wn, btype="bandpass", output="sos")
        filters.append(sos)
    return filters

FILTER_BANK = design_filter_bank()
FB_WEIGHTS = np.asarray([(m + 1) ** (-1.25) + 0.25 for m in range(NUM_FILTER_BANKS)], dtype=np.float64)
print("filter banks:", len(FILTER_BANK), "weights:", FB_WEIGHTS)


def predict_fbcca(x_test: np.ndarray) -> np.ndarray:
    preds = []
    for epoch in x_test:
        total = np.zeros(N_CLASSES, dtype=np.float64)
        for weight, sos in zip(FB_WEIGHTS, FILTER_BANK):
            filtered = signal.sosfiltfilt(sos, epoch[:, :WINDOW_SAMPLES], axis=-1)
            rho = cca_score_one(filtered)
            total += weight * (rho ** 2)
        preds.append(int(np.argmax(total)))
    return np.asarray(preds, dtype=int)


In [ ]:
def _regularized_eigh(sb: np.ndarray, sw: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    sb = (sb + sb.T) / 2.0
    sw = (sw + sw.T) / 2.0
    eps = 1e-6 * np.trace(sw) / max(sw.shape[0], 1)
    if not np.isfinite(eps) or eps <= 0:
        eps = 1e-6
    sw = sw + eps * np.eye(sw.shape[0])
    values, vectors = eigh(sb, sw, check_finite=False)
    order = np.argsort(values)[::-1]
    return values[order], vectors[:, order]


def tdca_augment(trials: np.ndarray, projector: np.ndarray, delay_points: int = DELAY_POINTS) -> np.ndarray:
    """TDCA augmented signal: delay embedding + projection-enhanced copy."""
    trials = np.asarray(trials, dtype=np.float64)
    if trials.ndim == 2:
        trials = trials[np.newaxis, ...]
    n_trials, n_channels, n_points = trials.shape
    n_samples = projector.shape[0]
    if n_points < n_samples + delay_points:
        raise ValueError(f"TDCA needs {n_samples + delay_points} samples, got {n_points}")

    delayed = np.empty((n_trials, n_channels * (delay_points + 1), n_samples), dtype=np.float64)
    for lag in range(delay_points + 1):
        delayed[:, lag * n_channels:(lag + 1) * n_channels, :] = trials[:, :, lag:lag + n_samples]
    delayed = delayed - delayed.mean(axis=-1, keepdims=True)
    projected = delayed @ projector
    return np.concatenate([delayed, projected], axis=-1)


def dsp_fit(aug_x: np.ndarray, y: np.ndarray, n_components: int = TDCA_COMPONENTS) -> tuple[np.ndarray, np.ndarray]:
    aug_x = aug_x - aug_x.mean(axis=-1, keepdims=True)
    labels = np.unique(y)
    grand_mean = aug_x.mean(axis=0)
    n_features = aug_x.shape[1]
    sw = np.zeros((n_features, n_features), dtype=np.float64)
    sb = np.zeros((n_features, n_features), dtype=np.float64)

    for label in labels:
        xi = aug_x[y == label]
        mi = xi.mean(axis=0)
        centered = xi - mi
        sw += np.einsum("nct,ndt->cd", centered, centered, optimize=True)
        diff = mi - grand_mean
        sb += xi.shape[0] * (diff @ diff.T)

    _, w = _regularized_eigh(sb, sw)
    w = w[:, :n_components]
    features = np.einsum("fc,nct->nft", w.T, aug_x, optimize=True)
    templates = np.zeros((N_CLASSES, n_components, aug_x.shape[-1]), dtype=np.float64)
    for label in range(N_CLASSES):
        templates[label] = features[y == label].mean(axis=0)
    return w, templates


def corr_flat(a: np.ndarray, b: np.ndarray) -> float:
    a = np.ravel(a).astype(np.float64)
    b = np.ravel(b).astype(np.float64)
    a -= a.mean()
    b -= b.mean()
    denom = math.sqrt(float(a @ a) * float(b @ b))
    if denom <= 1e-12:
        return 0.0
    return float((a @ b) / denom)


@dataclass
class FBTDCA:
    n_bands: int = NUM_FILTER_BANKS
    delay_points: int = DELAY_POINTS
    n_components: int = TDCA_COMPONENTS

    def fit(self, x_train: np.ndarray, y_train: np.ndarray):
        self.models_ = []
        self.filters_ = FILTER_BANK[: self.n_bands]
        self.weights_ = FB_WEIGHTS[: self.n_bands]
        for sos in self.filters_:
            xf = signal.sosfiltfilt(sos, x_train, axis=-1)
            band_aug, band_y = [], []
            for label in range(N_CLASSES):
                xi = xf[y_train == label]
                aug = tdca_augment(xi, PROJECTORS[label], self.delay_points)
                band_aug.append(aug)
                band_y.append(np.full(xi.shape[0], label, dtype=int))
            aug_x = np.concatenate(band_aug, axis=0)
            aug_y = np.concatenate(band_y, axis=0)
            w, templates = dsp_fit(aug_x, aug_y, self.n_components)
            self.models_.append((w, templates))
        return self

    def decision_function(self, x_test: np.ndarray) -> np.ndarray:
        x_test = np.asarray(x_test, dtype=np.float64)
        if x_test.ndim == 2:
            x_test = x_test[np.newaxis, ...]
        scores_all = np.zeros((x_test.shape[0], N_CLASSES), dtype=np.float64)
        for weight, sos, (w, templates) in zip(self.weights_, self.filters_, self.models_):
            xf = signal.sosfiltfilt(sos, x_test, axis=-1)
            for trial_i in range(xf.shape[0]):
                class_scores = np.zeros(N_CLASSES, dtype=np.float64)
                for label in range(N_CLASSES):
                    aug = tdca_augment(xf[trial_i], PROJECTORS[label], self.delay_points)[0]
                    feat = w.T @ aug
                    class_scores[label] = corr_flat(feat, templates[label])
                scores_all[trial_i] += weight * class_scores
        return scores_all

    def predict(self, x_test: np.ndarray) -> np.ndarray:
        return np.argmax(self.decision_function(x_test), axis=1).astype(int)


In [ ]:
def accuracy(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.mean(np.asarray(y_true) == np.asarray(y_pred)))


def evaluate_subject(sid: int, mat_path: Path) -> tuple[list[dict], list[dict]]:
    data = load_subject_data(mat_path)
    x_full, y, block_ids = extract_trials(data, extra_points=DELAY_POINTS)
    blocks = np.unique(block_ids)
    block_rows, trial_rows = [], []

    print(f"S{sid:02d}: x={x_full.shape}, blocks={blocks.tolist()}")
    for test_block in blocks:
        train_idx = block_ids != test_block
        test_idx = block_ids == test_block
        x_train, y_train = x_full[train_idx], y[train_idx]
        x_test, y_test = x_full[test_idx], y[test_idx]

        methods = []
        methods.append(("CCA", predict_cca(x_test[:, :, :WINDOW_SAMPLES])))
        if RUN_FBCCA:
            methods.append(("FBCCA", predict_fbcca(x_test[:, :, :WINDOW_SAMPLES])))
        if RUN_FBTDCA:
            model = FBTDCA().fit(x_train, y_train)
            methods.append(("FBTDCA", model.predict(x_test)))

        for method, pred in methods:
            acc = accuracy(y_test, pred)
            block_rows.append({"subject": sid, "block": int(test_block), "method": method, "accuracy": acc})
            for yt, yp in zip(y_test, pred):
                trial_rows.append({
                    "subject": sid,
                    "block": int(test_block),
                    "method": method,
                    "true_label": int(yt),
                    "pred_label": int(yp),
                    "true_freq": float(TARGET_FREQS[int(yt)]),
                    "pred_freq": float(TARGET_FREQS[int(yp)]),
                    "correct": int(yt == yp),
                })
            print(f"  block {int(test_block) + 1}/{len(blocks)} {method:6s}: {acc:.3f}")
    return block_rows, trial_rows


all_block_rows, all_trial_rows = [], []
for sid, path in subject_files:
    b_rows, t_rows = evaluate_subject(sid, path)
    all_block_rows.extend(b_rows)
    all_trial_rows.extend(t_rows)

block_df = pd.DataFrame(all_block_rows)
trial_df = pd.DataFrame(all_trial_rows)
subject_df = (
    block_df.groupby(["subject", "method"], as_index=False)["accuracy"]
    .mean()
    .sort_values(["subject", "method"])
)
summary_df = (
    subject_df.groupby("method", as_index=False)["accuracy"]
    .agg(["mean", "std", "count"])
    .reset_index()
)

block_df.to_csv(RESULT_DIR / "block_summary.csv", index=False, encoding="utf-8-sig")
trial_df.to_csv(RESULT_DIR / "trial_predictions.csv", index=False, encoding="utf-8-sig")
subject_df.to_csv(RESULT_DIR / "subject_summary.csv", index=False, encoding="utf-8-sig")
summary_df.to_csv(RESULT_DIR / "method_summary.csv", index=False, encoding="utf-8-sig")

print("\nSubject summary")
display(subject_df)
print("\nMethod summary")
display(summary_df)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.8), dpi=130)
plot_df = subject_df.groupby("method", as_index=False)["accuracy"].mean()
ax.bar(plot_df["method"], plot_df["accuracy"], color=["#4C78A8", "#F58518", "#54A24B"][: len(plot_df)])
ax.set_ylim(0, 1.0)
ax.set_ylabel("Mean accuracy")
ax.set_xlabel("Method")
ax.set_title(f"Benchmark SSVEP {WINDOW_SEC:.1f}s window")
ax.grid(axis="y", alpha=0.25)
for i, row in plot_df.iterrows():
    ax.text(i, row["accuracy"] + 0.02, f"{row['accuracy']:.3f}", ha="center", va="bottom")
fig.tight_layout()
fig.savefig(RESULT_DIR / "algorithm_accuracy.png")
plt.show()

print("Saved files:")
for path in sorted(RESULT_DIR.glob("*")):
    print(" -", path)


## ????

?? S1 ???????????????????

```python
SUBJECT_LIMIT = None
RUN_FBCCA = True
RUN_FBTDCA = True
```

?? 35 ???? FBTDCA ????????? `SUBJECT_LIMIT = 1` ? `3`???????????
